In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 15.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://amigo-capable-untying.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://amigo-capable-untying.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload) #記住過往詢問的訊息，再來延續上個訊息來回應
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, 簡稱明新科大）是一所位於**臺灣新竹縣新豐鄉**的私立科技大學。

以下是其主要簡介：

1.  **創立與發展：** 創立於1966年，前身為明新工業專科學校，後於1997年改制為明新技術學院，並於2002年升格為科技大學。
2.  **地理位置優勢：** 學校鄰近**新竹科學園區**，使其在產學合作、人才培育與產業接軌方面具有得天獨厚的優勢，與高科技產業的連結深厚。
3.  **學院與系所：** 學校設有多個學院，主要涵蓋：
    *   **工程學院**
    *   **管理學院**
    *   **服務事業學院** (特色學院，含觀光、旅館、餐飲等)
    *   **設計學院**
    提供學士、碩士等學位課程。
4.  **教學特色：**
    *   **實務導向：** 強調理論與實務結合，注重學生動手實作能力，培養符合產業需求的專業技術人才。
    *   **產學合作：** 積極與企業界合作，提供實習機會，提高學生的就業競爭力。
    *   **全人教育：** 除了專業知識外，也重視學生的人文素養、品德教育與國際視野。
5.  **辦學目標：** 以培育具備「專業知能、人文素養、國際視野」的優質專業人才為目標。

總體而言，明新科技大學是一所深耕台灣產業、注重實務應用與學生就業競爭力的科技型大學，尤其在新竹地區的產業人才供應鏈中扮演重要角色。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長是 **劉國偉 教授**。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615031124456112512","quoteToken":"MMvn2AP1Li3lj2zdWAkwbG6t-GJn8AkI1eN1ZFY9PborgOPPdheGoSSPEMKko_Vng2grvaQcNlwcPtLD7OoE0tAZKjzYBE8CWcUlDCUVGajDEm9HOI5bbmoGT42YjXveAqUmdekVB1ME7TxNDpDjSw","markAsReadToken":"Llapb6L_zPCw8AGsvdIIy_QGZ7LdViO6TiDqZ22jD6PcKMd_7gFbcihf8iA53dis0Q2yNs9NtTKX7T0w75BWRJ71e5sJsnzUqhbbHUnd388QClKQAKFBOe_kiHoHguL9NwtCwHptr433atnbLrXU9qHDFCTT8iNtb3MO2mj3KP9mfV_Gy_IjlFFKeLlHgmx-PW-7ir9XD1zNLg1KXm4RcA","text":"AI 請用兩點來簡介資訊管理系，限50-60字內"},"webhookEventId":"01KS6SMP1FG3YAYJCJN4RYWFQZ","deliveryContext":{"isRedelivery":false},"timestamp":1779418421062,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"c53d4395b4834e328b5afb0ef25527b3","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:53:47] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615031205624545760","quoteToken":"2zhR6Secnp0VJfvGXb23TVYB51LJxb0bUyrNRHTNAEO6HP0QGVwuO-qiAXAwNvi9AkcJqA9fChLhLCkiFpdfzuh1cdOOmIshLyXlDe2M_Ek0XI6xlZ9vEFux8tMOUoaluJlTEn3D5bhaLILR9WwR-g","markAsReadToken":"fFx3k_99olngGNeYR5ioYW0Ez9Zc57d5JcyZ_Ti8ikhCAKX_JboEUWUXHR52--7AggMxqVh-_niznIj4fAplm3gmN9Mjjf-HRTF5haKgbC5Br9Dk5aWZwv4gI-GBEuHWhfphStBe0FTQzyMmgNw4mZmav_QKV7HvPXChA5H7EuKVrqtyG_Aa0bPIivOm1EapBW2LBMTryBX9LwKENVvEyg","text":"AI 那來講解優缺點各兩點，限70字內"},"webhookEventId":"01KS6SP5G3FWBAANE8M3SQGWXS","deliveryContext":{"isRedelivery":false},"timestamp":1779418469447,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"4cdede0e81fd442c889040af71600ba2","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:54:39] "POST / HTTP/1.1" 200 -
